In [3]:
import spacy
import numpy as np

from conlleval import evaluate

tokens = []
etiquetas = []
etiquetas_pred = []
tokens_etiquetados = []

# Carga del archivo
with open('./esp.testb', 'r') as corpus:
	for linea in corpus:
		# Se elimina los carácteres blancos de los extremos
		linea = linea.strip()

		if not linea:
			continue 

		division_linea = linea.split(' ', maxsplit=1)

		tokens.append(division_linea[0])
		etiquetas.append(division_linea[1])

# Creación del texto en plano
texto_plano_espaciado = " ".join(tokens)

# Uso del modelo
nlp = spacy.load("es_core_news_lg")

doc = nlp(texto_plano_espaciado)

# Obtención del etiquetado
for token in doc:
	tokens_etiquetados.append(token.text)
	etiquetas_pred.append(f"{token.ent_iob_}{f'-{token.ent_type_}' if token.ent_type_ else ''}")

evaluate(etiquetas, etiquetas_pred)

processed 51533 tokens with 3558 phrases; found: 4008 phrases; correct: 94.
accuracy:   3.76%; (non-O)
accuracy:  75.01%; precision:   2.35%; recall:   2.64%; FB1:   2.48
              LOC: precision:   2.58%; recall:   3.41%; FB1:   2.94  1434
             MISC: precision:   0.65%; recall:   1.47%; FB1:   0.90  767
              ORG: precision:   3.89%; recall:   2.93%; FB1:   3.34  1053
              PER: precision:   1.46%; recall:   1.50%; FB1:   1.48  754


(2.345309381237525, 2.641933670601462, 2.4848004229447525)

----

In [4]:
print(len(etiquetas), len(etiquetas_pred))

51533 51761


----

In [5]:
for ind in range(0,len(etiquetas_pred)):
	if tokens_etiquetados[ind] != tokens[ind]:
		print(tokens[ind], tokens_etiquetados[ind], tokens_etiquetados[ind+1])
		break

20. 20 .


----

In [ ]:
import re

tokens = []
etiquetas = []
etiquetas_pred = []
tokens_etiquetados = []

# Carga del archivo
with open('./esp.testb', 'r') as corpus:
	for linea in corpus:
		# Se elimina los carácteres blancos de los extremos
		linea = linea.strip()

		if not linea:
			continue 

		division_linea = linea.split(' ', maxsplit=1)
		token = division_linea[0]
		etiqueta = division_linea[1]

		# Si {número}[. o ,]{número}
		if re.search(r'\d+[,\.]\d+', token):
			token = token.replace('.', '')
			token = token.replace(',', '.')

			tokens.append(token)
			etiquetas.append(etiqueta)
		
		# Si ({número}[.])
		# 	o ({texto}[, o . o :] y etiqueta es O o I-xxx)
		# 	o ({texto}/)
		elif (
				re.search(r'\d+\.', token) or (
					re.search(r'[a-zA-Z]+[,\.:]$', token) and 
					(etiqueta == 'O' or etiqueta.startswith('I'))
				) or
				re.search(r'[a-zA-Z]/$', token)
			):
			tokens.append(token[:-1])
			tokens.append(token[-1])
			etiquetas.append(etiqueta)
			etiquetas.append(etiqueta)

		# Si {texto}[, o . o :] y etiqueta es B-xxx (no hace falta comprobación)
		elif re.search(r'[a-zA-Z]+[,\.:]$', token):
			tokens.append(token[:-1])
			tokens.append(token[-1])

			categoria = etiqueta.split('-')[1]
			etiquetas.append(f"B-{categoria}")
			etiquetas.append(f"I-{categoria}")

		# Si {texto}/{texto}
		elif re.search(r'[a-zA-Z]+\/[a-zA-Z]+$', token):
			div_token = token.split('/')

			tokens.append(div_token[0])
			tokens.append('/')
			tokens.append(div_token[1])

			if etiqueta == 'O' or etiqueta.startswith('I'):
				etiquetas = etiquetas + [etiqueta] * 3
			else:
				categoria = etiqueta.split('-')[1]
				etiquetas.append(f"B-{categoria}")
				etiquetas = etiquetas + [f"I-{categoria}"] * 2

		# caso normal
		else:
			tokens.append(token)
			etiquetas.append(etiqueta)

# Creación del texto en plano
texto_plano_espaciado = " ".join(tokens)

# Uso del modelo
nlp = spacy.load("es_core_news_lg")

doc = nlp(texto_plano_espaciado)

# Obtención del etiquetado
for token in doc:
	tokens_etiquetados.append(token.text)
	etiquetas_pred.append(f"{token.ent_iob_}{f'-{token.ent_type_}' if token.ent_type_ else ''}")

evaluate(etiquetas, etiquetas_pred)

processed 51749 tokens with 3558 phrases; found: 4017 phrases; correct: 828.
accuracy:  24.35%; (non-O)
accuracy:  80.32%; precision:  20.61%; recall:  23.27%; FB1:  21.86
              LOC: precision:  19.11%; recall:  25.37%; FB1:  21.80  1439
             MISC: precision:   5.30%; recall:  12.09%; FB1:   7.37  773
              ORG: precision:  27.54%; recall:  20.71%; FB1:  23.64  1053
              PER: precision:  29.52%; recall:  30.20%; FB1:  29.86  752


(20.612397311426438, 23.27150084317032, 21.861386138613863)

In [45]:
print(len(etiquetas), len(etiquetas_pred))

51749 51835


In [46]:
for ind in range(0,len(etiquetas_pred)):
	if tokens_etiquetados[ind] != tokens[ind]:
		print(tokens[ind], tokens_etiquetados[ind], tokens_etiquetados[ind+1])
		break

Const.. Const ..


----

In [40]:
with open('corpus_tmp.txt', 'w') as fp:
	for ind in range(0,len(etiquetas)):
		fp.write(f"{tokens[ind]} {etiquetas[ind]}\n")

In [44]:
with open('salida_tmp.txt', 'w') as fp:
	for ind in range(0,len(tokens_etiquetados)):
		fp.write(f"{tokens_etiquetados[ind]} {etiquetas_pred[ind]}\n")